# 🎯 [K06_M04] 요약 결과 확인하고 바로잡기
### 실습 솔루션 & 정답 해설 (Google Colab / Jupyter)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/K06/k06_m04_solution.ipynb)

---

## 📌 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 / 과제** | 일반행정·부대관리 (Level 1) / K06 군 공지 사항 요약 AI |
| **미션** | 미션 04. 요약 결과 확인하고 바로잡기 (소요시간: 약 1.5시간) |
| **실습 유형** | 생성형 AI 프롬프트 엔지니어링 및 팩트체크 검증 |

> **⚠️ [교육용 가상 자료 원칙]**
> 본 실습의 모든 자료는 안전 규정에 따라 **교육용 가상 부대 자료(가상 한빛대대)**로 구성되어 있습니다.
> 실제 군사 기밀이나 부대 세부 인적·작전 정보는 외부 AI 도구에 절대 입력하지 않습니다.



In [ ]:
# ============================================================================
# [환경 설정 및 헬퍼 함수] 마크다운 시각화 렌더러 및 가상 공지 데이터 뷰어
# ============================================================================
from IPython.display import display, Markdown
import json
import os

# 교육용 가상 공지 데이터베이스 (N01 ~ N12, 추가 공지 A1 ~ A2)
NOTICES_DB = {
    "N01": {"date": "11-02", "dept": "대대 행정과", "title": "11월 정기 휴가 신청 접수 안내", "content": "11월 정기 휴가 신청을 11월 5일(목) 18시까지 접수합니다. 신청은 중대 행정반에 비치된 휴가 신청서를 작성하여 소속 분대장에게 제출합니다. 휴가 기간이 겹치는 인원이 중대 정원의 20%를 넘으면 선착순이 아닌 추첨으로 조정합니다. 결과는 11월 9일(월) 중대 게시판에 게시합니다. 문의: 행정과 담당관(내선 1101)."},
    "N02": {"date": "11-02", "dept": "대대 의무실", "title": "독감 예방접종 일정 안내", "content": "독감 예방접종을 11월 10일(화)부터 11월 12일(목)까지 중대별로 실시합니다. 1중대는 10일 09시, 2중대는 11일 09시, 3중대는 12일 09시에 의무실 앞으로 집합합니다. 접종 당일 발열이 있는 인원은 접종 전 의무병에게 알리고, 접종 후 30분간 의무실 대기 후 복귀합니다. 미접종자는 11월 16일(월) 추가 접종을 받습니다."},
    "N03": {"date": "11-03", "dept": "대대 군수과", "title": "동계 피복 보급 일정", "content": "동계 피복(방한 내피, 방한 장갑, 넥워머)을 11월 6일(금) 13시부터 17시까지 중대 순서대로 보급합니다. 분대장은 분대원 치수표를 11월 4일(수)까지 군수과에 제출합니다. 치수 교환은 보급 당일에만 가능하며, 이후에는 다음 달 정기 보급 때 교환합니다."},
    "N04": {"date": "11-03", "dept": "대대 정보통신실", "title": "사이버 보안 점검 및 비밀번호 변경", "content": "11월 9일(월)부터 13일(금)까지 사이버 보안 점검을 실시합니다. 모든 인원은 11월 8일(일)까지 업무용 계정 비밀번호를 변경해야 합니다. 비밀번호는 영문·숫자·특수문자를 포함해 9자리 이상으로 설정하고, 이전 비밀번호와 같으면 안 됩니다. 개인 휴대전화로 업무 자료를 촬영하거나 전송하는 행위는 금지됩니다."},
    "N05": {"date": "11-04", "dept": "대대 교육훈련과", "title": "11월 사격 훈련 일정 변경", "content": "기상 악화 예보로 11월 6일(금) 예정이던 2중대 사격 훈련을 11월 13일(금)로 연기합니다. 1중대와 3중대 일정은 변경 없습니다. 2중대는 6일에 실내 교육(사격술 예비훈련)으로 대체하며, 장소는 대대 강당입니다. 세부 시간표는 중대장 지시에 따릅니다."},
    "N06": {"date": "11-04", "dept": "대대 복지과", "title": "부대 내 복지시설 운영 시간 조정", "content": "동절기를 맞아 11월 9일(월)부터 체력단련실 운영 시간을 06시~21시로, 사이버지식정보방 운영 시간을 18시~21시 30분으로 조정합니다. 매주 수요일 14시~16시는 체력단련실 소독으로 이용할 수 없습니다. 복지매장(PX) 운영 시간은 기존과 같습니다."},
    "N07": {"date": "11-05", "dept": "대대 인사과", "title": "자기개발 지원 신청 안내", "content": "2027년 상반기 자기개발 비용 지원 신청을 11월 20일(금)까지 받습니다. 지원 대상은 자격증 응시료와 온라인 강좌 수강료이며, 1인당 연간 한도 내에서 실비를 지원합니다. 신청 시 응시 확인서 또는 수강 등록 확인서를 첨부합니다. 지원 여부는 12월 첫째 주에 개별 통보합니다."},
    "N08": {"date": "11-05", "dept": "대대 행정과", "title": "11월 정기 휴가 신청 접수 기한 연장", "content": "11월 정기 휴가 신청 접수 기한을 11월 5일(목) 18시에서 11월 6일(금) 12시로 연장합니다. 사격 훈련 일정 변경에 따른 조치입니다. 결과 게시일(11월 9일)은 변경 없습니다."},
    "N09": {"date": "11-06", "dept": "대대 안전관리관", "title": "동절기 안전사고 예방 수칙", "content": "동절기 안전사고 예방을 위해 다음 사항을 지킵니다. 생활관 내 개인 전열기 사용을 금지합니다. 차량 운행 전 타이어 상태와 결빙 구간을 확인합니다. 야외 활동 시 방한 장구를 착용하고 2인 1조로 움직입니다. 동상 의심 증상이 있으면 즉시 의무실에 알립니다."},
    "N10": {"date": "11-06", "dept": "대대 의무실", "title": "독감 예방접종 3중대 일정 변경", "content": "3중대 독감 예방접종을 11월 12일(목) 09시에서 11월 12일(목) 14시로 변경합니다. 오전 훈련 일정 조정에 따른 것입니다. 1중대와 2중대 일정은 변경 없습니다."},
    "N11": {"date": "11-09", "dept": "대대 군수과", "title": "생활관 난방기 점검", "content": "11월 11일(수)부터 13일(금)까지 생활관 난방기를 점검합니다. 점검 시간에는 생활관을 비워 두고, 귀중품은 개인 관물대에 보관합니다. 난방기 이상 소음이나 냄새가 있으면 점검 전까지 중대 행정반에 신고합니다."},
    "N12": {"date": "11-09", "dept": "대대 교육훈련과", "title": "AI 활용 교육 참가자 모집", "content": "대대 자체 AI 활용 교육 참가자를 모집합니다. 교육은 11월 23일(월)부터 4주간 매주 월요일 19시~21시에 사이버지식정보방에서 진행합니다. 중대별 5명 이내로 11월 16일(월)까지 중대 행정반에 신청합니다. 부대 자료를 외부 AI 서비스에 입력하지 않는 보안 수칙 교육이 포함됩니다."},
    "A1": {"date": "11-10", "dept": "대대 복지과", "title": "동계 체육대회 참가 신청", "content": "11월 27일(금) 대대 체육대회를 실시합니다. 종목은 축구, 족구, 줄다리기이며 중대별 참가 명단을 11월 18일(수)까지 복지과에 제출합니다. 우천 시 11월 30일(월)로 연기합니다. 문의: 복지과(내선 1503)."},
    "A2": {"date": "11-10", "dept": "대대 정보통신실", "title": "업무용 PC 보안 업데이트", "content": "11월 12일(목) 18시부터 20시까지 업무용 PC 보안 업데이트를 진행합니다. 업데이트 중에는 PC를 끄지 말고, 18시 전까지 작업 중인 문서를 저장합니다. 업데이트 후 오류가 있으면 정보통신실(내선 1401)에 신고합니다."}
}

def view_notice(notice_id):
    """가상 공지 원문을 편리하게 확인하는 함수"""
    if notice_id in NOTICES_DB:
        n = NOTICES_DB[notice_id]
        print(f"[{notice_id}] {n['title']} (작성: {n['date']} / {n['dept']})")
        print(f"본문: {n['content']}")
    else:
        print(f"존재하지 않는 공지 ID입니다: {notice_id}")

def show_result(mission_title, prompt, result):
    """학습자가 입력한 프롬프트와 AI 요약 결과를 시각적으로 렌더링하는 헬퍼 함수"""
    p_text = prompt.strip() if prompt.strip() else "(작성된 프롬프트가 없습니다)"
    r_text = result.strip() if result.strip() else "(기록된 결과가 없습니다)"
    
    md_content = f"""
---
### 📌 [{mission_title}] 실습 기록 리포트

#### 💬 입력 프롬프트
```text
{p_text}
```

#### 📋 AI 요약 결과 및 팩트체크 검증 내역
{r_text}
---
"""
    display(Markdown(md_content))

print("✅ 워크북 시각화 헬퍼 함수 및 가상 공지 데이터베이스(N01~N12, A1~A2) 준비 완료!")



---

## 🎯 미션 04. 요약 결과 확인하고 바로잡기
**소요시간**: 약 1.5시간

### 📋 미션 배경 및 지침
#### 미션 배경 및 당위성

AI가 생성한 요약문을 원문과 문장 단위로 1:1 대조하여 오류, 왜곡, 환각(Hallucination) 현상을 식별합니다. 발견된 오류를 구체적으로 지적하는 교정 프롬프트를 작성하여 AI에게 재요청하고 무결점 요약본을 완성합니다.

**이번 미션 결과물:** `요약-원문 1:1 교차 대조표 및 교정 완료본` 완성

### 📑 실습 서식 및 대조 점검 양식
| 요약문 문장 | 원문 근거 문장 | 오류 / 왜곡 / 환각 여부 | 교정 내용 |
| :--- | :--- | :--- | :--- |
| 1행: | | [ ] 정상 / [ ] 오류 | |
| 2행: | | [ ] 정상 / [ ] 오류 | |
| 3행: | | [ ] 정상 / [ ] 오류 | |

### 🔍 평가 기준 (PASS 조건)
#### 미션 판정
- **평가 구분:** 필수 ([필수])
- **핵심 평가 기준:** [필수] 대조 점검 — 요약과 원문을 1:1 대조해 오류나 빠진 부분을 찾아 고쳤는가?
- **미션 PASS 조건:**
  1. 요구 산출물의 완성도 충족
  2. 원문 대조 점검을 통한 사실 검증 및 정확성 확보 (날짜·시간·숫자 무결점)
  3. 교육용 가상 데이터 활용 및 군사보안 수칙 엄수



In [ ]:
# [미션 04. 요약 결과 확인하고 바로잡기] 프롬프트 및 실습 결과 기록
mission_04_prompt = """# [오류 교정 피드백 프롬프트]
앞서 작성해 준 요약문에서 다음 2가지 중요 행정 정보가 누락되었습니다.
1. '중대 정원의 20% 초과 시 선착순이 아닌 추첨 조정' 규칙
2. '결과 게시일: 11월 9일(월) 중대 게시판'

원문의 날짜와 숫자를 왜곡하지 말고, 위 2가지 정보를 반드시 포함하여 정확한 3개 항목으로 다시 요약해 줘.
"""

mission_04_result = """### 📋 1. 요약-원문 1:1 교차 대조표 (Fact-checking Sheet)

| 요약문 항목 | 원문 근거 문장 | 오류 / 누락 / 환각 여부 | 교정 조치 내용 |
| :--- | :--- | :---: | :--- |
| 1. 11월 5일(목) 18시까지 신청 접수 | 11월 정기 휴가 신청을 11월 5일(목) 18시까지 접수합니다. | [정상] 일치 | 날짜·시간 원문과 100% 일치 |
| 2. 분대장에게 신청서 제출 | 신청은 중대 행정반에 비치된 휴가 신청서를 작성하여 소속 분대장에게 제출합니다. | [정상] 일치 | 행정반 비치 서식 및 분대장 제출 유지 |
| (누락 항목 1) | 휴가 기간이 겹치는 인원이 중대 정원의 20%를 넘으면 선착순이 아닌 추첨으로 조정합니다. | **[치명적 누락]** | 분대원 민원 방지를 위해 '20% 초과시 추첨' 항목 필수 삽입 |
| (누락 항목 2) | 결과는 11월 9일(월) 중대 게시판에 게시합니다. | **[누락]** | 발표일(11/9) 및 확인 장소(게시판) 추가 |

---
### ✅ 2. 피드백 후 재생성된 무결점 요약 결과
1. **[신청 기한 및 방법]**: 11월 5일(목) 18시까지 중대 행정반의 신청서를 작성하여 소속 분대장에게 제출.
2. **[선발 기준 및 결과]**: 중대 정원 20% 초과 시 추첨으로 조정하며, 결과는 11월 9일(월) 중대 게시판에 공지.
3. **[문의처]**: 대대 행정과 담당관 (내선 1101).
"""

# 렌더러 실행 (결과를 시각적으로 확인합니다)
show_result("미션 04. 요약 결과 확인하고 바로잡기", mission_04_prompt, mission_04_result)

